# Lab 03｜文件在磁盘上，模型就已经知道了吗？

先只交给 Agent 告警，让它列出已知事实；再在同一 Thread 读取部署记录，回答同一个问题。对照实际请求、工具回执和两次回答，观察文件内容从什么时候开始影响判断。

## 核心代码：同一问题，读取前后作对照

以下代码摘自第 1 节，仅供定位；运行仍从公共准备开始。

```python
before_start = len(lab.requests)
before = run(thread, alert + "\n" + question + "\n不要读取文件或调用工具。")
start = len(lab.requests)
result = run(thread, "先读取 deployment.md，再回答：" + question)
```

两次运行使用同一 Thread 和同一个问题。`before_start`、`start` 区分各次请求，后续表格核对部署记录是否通过工具结果进入输入。文件存在不等于模型已经收到文件正文。


## 公共准备｜必跑，代码可折叠

选择 **Python 3.12** 内核，保留仓库的 `examples/notebook_support.py`。下格会自动补装固定版本的实验依赖；如果当前内核已经导入其他版本，请安装后重启内核。无需先到 README 手动安装。

九讲统一读取 `GLM_API_KEY`；未设置时弹出隐藏输入框。模型由 GLM-5.2 提供，不需要 OpenAI / ChatGPT 账号；开始任务后会产生 API 费用。准备区只启动本地连接，不调用模型。每讲可从本格独立启动，最后一格关闭连接；再次从头运行前先执行清理。

版本检查、协议适配、健康检查和退出处理属于辅助代码，不必逐行阅读。主流程以“必读必跑”标注；“扩展/可选”默认跳过。


In [ ]:
import sys, os, json, secrets, time, subprocess, getpass
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError
from IPython.display import display, Markdown, HTML
from html import escape
from contextlib import redirect_stdout
from io import StringIO

if sys.version_info[:2] != (3, 12):
    raise RuntimeError("请选择 Python 3.12 内核，再从头运行。")
required = {"openai-codex": "0.154.0", "litellm": "1.101.0"}
missing = []
for package, expected in required.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    if installed != expected:
        missing.append(("litellm[proxy]" if package == "litellm" else package) + "==" + expected)
if missing:
    already_loaded = any(name in sys.modules for name in ("openai_codex", "litellm"))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
    if already_loaded:
        raise RuntimeError("依赖已安装；请重启内核并重新运行，避免混用旧模块。")

here = Path.cwd().resolve()
examples = next((p for base in (here, *here.parents)
                 for p in (base, base / "examples") if (p / "notebook_support.py").is_file()), None)
if examples is None:
    raise FileNotFoundError("请在完整仓库内打开本 notebook。")
sys.path.insert(0, str(examples))
from notebook_support import Lab, command_receipts, tool_output_contains
from openai_codex import ApprovalMode, Sandbox
if not os.environ.get("GLM_API_KEY", "").strip():
    os.environ["GLM_API_KEY"] = getpass.getpass("GLM_API_KEY：").strip()
if "lab" in globals() and not lab._closed:
    raise RuntimeError("本讲连接仍在运行，请先执行最后的 lab.close()。")
lab = Lab(examples / "03-context-engine")

# 展示函数只整理真实材料，不把预期结果冒充本次运行结果。
def table(headers, rows):
    head = "".join(f"<th>{escape(str(v))}</th>" for v in headers)
    body = "".join("<tr>" + "".join(
        "<td><pre style='white-space:pre-wrap;max-width:52em'>" + escape(lab.clean(v)) + "</pre></td>"
        for v in row) + "</tr>" for row in rows)
    display(HTML(f"<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>"))

def details(title, value):
    display(HTML("<details><summary>" + escape(title) + "</summary><pre style='white-space:pre-wrap'>"
                 + escape(lab.clean(value)) + "</pre></details>"))

def run(thread, prompt, **kwargs):
    # 保留运行错误；仅收起辅助层重复打印的状态和回答。
    with redirect_stdout(StringIO()):
        return lab.run(thread, prompt, **kwargs)

def events(result):
    rows = []
    for wrapped in result.items:
        item = wrapped.root
        if item.type == "commandExecution":
            rows.extend([("采取的动作", item.command),
                         ("刚获得的信息", item.aggregated_output or "（无文本输出）")])
        elif item.type == "fileChange":
            rows.append(("文件操作", ", ".join(change.path for change in item.changes)))
        elif item.type == "agentMessage" and item.text.strip():
            rows.append(("Agent 的判断", item.text))
    table(["事件", "本次实际内容"], rows)

def snapshot(folder):
    return {str(p.relative_to(folder)):p.read_bytes() for p in folder.rglob("*") if p.is_file()}

def compare_inputs(folder, before):
    after = snapshot(folder)
    changed = [name for name in sorted(before.keys() | after.keys()) if before.get(name) != after.get(name)]
    print("输入文件：" + ("有变化，展开下方查看" if changed else "未改变"))
    for name in changed:
        details(name, "运行前：\n" + before.get(name,"[不存在]".encode()).decode(errors="replace")
                + "\n运行后：\n" + after.get(name,"[不存在]".encode()).decode(errors="replace"))



WORK = lab.work

def excerpt(text, needles, width=140):
    text = lab.clean(text)
    spans = []
    for needle in needles:
        position = text.find(needle)
        if position >= 0:
            spans.append((max(0, position-width), min(len(text),position+len(needle)+width)))
    if not spans:
        return text[:280] + ("…（展开原请求查看）" if len(text)>280 else "")
    merged = []
    for a,b in sorted(spans):
        if merged and a <= merged[-1][1]:
            merged[-1] = (merged[-1][0], max(b,merged[-1][1]))
        else:
            merged.append((a,b))
    return "\n…\n".join(text[a:b] for a,b in merged)

def show_requests(start=0):
    rows = []
    for i,request in enumerate(lab.requests[start:], start+1):
        if request.get("instructions"):
            rows.append((i,"固定指令",excerpt(str(request["instructions"]),[rule])))
        rows.append((i,"工具定义",", ".join(t.get("name",t.get("type","")) for t in request.get("tools",[]))))
        for item in request.get("input",[]):
            if not isinstance(item,dict):continue
            text = json.dumps(item,ensure_ascii=False)
            kind = "工具结果" if item.get("type")=="function_call_output" else item.get("role",item.get("type",""))
            rows.append((i,kind,excerpt(text,[rule, "读取 deployment.md",marker,"02:07","连接池上限"])))
    table(["请求", "组成", "规则、任务或编号附近的真实内容"],rows)
    details("扩展：完整请求（不含认证头）",json.dumps(lab.requests[start:],ensure_ascii=False,indent=2))

print("公共准备完成。")


## 1. 先只看告警，再读部署记录｜必读必跑

随机编号只写进部署文件，用于辨认工具结果；不要求读者抄写。第一次不读文件，只依据告警回答；第二次先读取部署记录，再回答同一个问题。


In [ ]:
import secrets
marker = "SAMPLE-" + secrets.token_hex(6)
(WORK / "deployment.md").write_text(
    f"采集编号：{marker}\n采集时间：02:07\n服务：payment-api\n"
    "部署记录：01:40 将数据库连接池上限由 200 下调到 60；是否引起告警仍待核对。\n", encoding="utf-8")
input_before = (WORK / "deployment.md").read_bytes()
rule = "区分已知事实和待验证的猜测；引用材料时说明文件名和时间，不要修改文件。"
alert = "02:07 payment-api P95 延迟超过 3 秒，5xx 错误率升至 8.4%。"
question = "目前已经确认哪些事实，还缺什么信息才能判断原因？"
thread = lab.thread(developer_instructions=rule)
before_start = len(lab.requests)
before = run(thread, alert + "\n" + question + "\n不要读取文件或调用工具。")
start = len(lab.requests)
result = run(thread, "先读取 deployment.md，再回答：" + question
             + "不要查询其他文件或网络；工具失败后停止，不重试。")
table(["同一问题", "本次实际回答"], [
    ("只根据告警，尚未读取部署记录", before.final_response),
    ("要求读取部署记录后", result.final_response),
])
events(result)


## 2. 打开每一次调用的工作现场

 这里记录的是 Codex 交给本地适配器的 Responses 请求体，尚未经过 LiteLLM 的协议转换。它能显示 Harness 的装配结果；不代表模型内部状态，也不显示模型的隐藏推理。
**必读必跑**。比较的是发送给模型的请求内容，不是两次回答的语气或准确程度。


In [ ]:
show_requests(before_start)
calls = lab.requests[start:]
observed_calls = lab.requests[before_start:]
table(["请求", "固定规则", "文件中的现场记录"], [
    (i + before_start, "已装入" if rule in json.dumps(r,ensure_ascii=False) else "未见",
     "已作为工具结果装入" if tool_output_contains(r,marker) else "尚未见工具结果")
    for i,r in enumerate(observed_calls,1)
])


## 3. 同一 Thread 再问一次｜必读必跑

这次输入不重述现场内容，也不要求再读文件。对照它的第一条请求，看前轮工具结果是否仍在。

**继续观察 / 可选**：把下格的 `RUN_NEW_THREAD` 改为 `True`，再比较新旧 Thread 的输入。扩展放在关闭连接之前；两次回答相似也不能证明看过相同材料。


In [ ]:
followup_start = len(lab.requests)
followup = run(thread, "根据刚才已经取得的记录，回答采集编号和采集时间。不要再调用工具。")
followup_request = lab.requests[followup_start]
old_receipts = {r["call_id"] for r in command_receipts(calls)}
new_receipts = {r["call_id"] for r in command_receipts(lab.requests[followup_start:])} - old_receipts
table(["位置", "工具结果中的记录", "本轮新工具调用"], [
    ("首次调用", "尚未读入" if not tool_output_contains(calls[0],marker) else "已读入", "见上表"),
    ("同一 Thread 追问", "仍在输入中" if tool_output_contains(followup_request,marker) else "未见",len(new_receipts)),
])
show_requests(followup_start)
compare_inputs(WORK, {"deployment.md":input_before})

# 扩展/可选：改为 True，在关闭连接前比较一个全新 Thread 的请求。
RUN_NEW_THREAD = False
if RUN_NEW_THREAD:
    fresh_start = len(lab.requests)
    fresh = run(lab.thread(), "根据刚才已经取得的记录，回答采集编号和采集时间。不要调用工具。")
    table(["Thread", "首次输入是否含现场编号"], [
        ("原 Thread", "含有" if marker in json.dumps(followup_request,ensure_ascii=False) else "没有"),
        ("全新 Thread", "含有" if marker in lab.request_text(fresh_start) else "没有"),
    ])


In [ ]:
# 观察和可选扩展结束后必跑；再次观察请从公共准备重新启动。
lab.close()
